# colab_10 — Cell-type annotation on the balanced 100k+100k integrated object

## Motivation

colab_04 annotated the original unbalanced 226k integrated object (Bhaduri 2020 + Zhong, before the Session 16 dataset swap). Its 19-cluster annotation is now stale because:
- The fetal partner changed (Zhong → Bhaduri 2021), introducing different RG/IPC/neuron representation.
- The composition is now balanced (100k+100k) instead of 224k+2.4k.
- colab_08's Leiden run produced 21 clusters at res=0.5, with cluster identities that don't map 1-to-1 to colab_04's 19.

This notebook annotates the 21 clusters of `integrated_100k_harmony.h5ad` (colab_08 output) and writes `integrated_annotated_100k.h5ad` for downstream trajectory work (colab_05 rerun) and any cross-dataset comparison.

## Approach

1. **Marker panels** — define per cell-type panels of canonical genes (RG, IPC, immature/mature excitatory, GABAergic, OPC, microglia, vascular, choroid plexus, stress).
2. **Per-cluster panel scoring** — mean expression of each panel within each Leiden cluster. **Caveat from colab_09 §5a**: panel-mean composites can be misleading when one gene in a panel dominates (KRT18/OTX2 made cluster 0 look like choroid plexus when it was actually cycling RG). So this is only a *first-pass summary*, never a decision rule.
3. **Unsupervised top markers** — `sc.tl.rank_genes_groups` (Wilcoxon) gives the data-driven top genes per cluster. This is the authoritative input for assignment.
4. **Dotplot of canonical markers across clusters** — visual confirmation.
5. **Cluster → cell-type mapping** — manual dict, filled in after inspecting §3/§4/§5 output.
6. **Apply, plot UMAPs by cell type, save.**

## Output

`integrated_annotated_100k.h5ad` on Drive at `data/processed/integrated/`. Adds `obs['cell_type']` (categorical) and an updated `obsm['X_umap']` (unchanged, just propagated).

Predicted size: ca. 6 GB (same as input).

## 0. Setup

### 0a — Install dependencies, mount Drive, imports

No Harmony / scanorama needed here — annotation is pure scanpy. Mounts Drive and defines `PATHS` for the input integrated object and the annotated output.

In [ ]:
!pip install -q scanpy leidenalg

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(6, 5))

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'
PATHS = {
    'integrated_in':  os.path.join(DRIVE_ROOT, 'data/processed/integrated/integrated_100k_harmony.h5ad'),
    'annotated_out':  os.path.join(DRIVE_ROOT, 'data/processed/integrated/integrated_annotated_100k.h5ad'),
}
print('scanpy', sc.__version__, '| anndata', ad.__version__)

## 1. Load the integrated object

### 1a — Read `integrated_100k_harmony.h5ad`

Loads the colab_08 output. Confirms shape (200000 × 2000 HVGs in `.X`, 16,768 in `.raw.X`), Leiden labels present, UMAP coords present.

In [ ]:
adata = sc.read_h5ad(PATHS['integrated_in'])

print(f'Shape: {adata.shape[0]:,} cells x {adata.shape[1]:,} HVGs')
print(f'.raw shape: {adata.raw.shape if adata.raw is not None else "None"}')
print(f'obsm keys: {list(adata.obsm.keys())}')
print(f'uns keys:  {list(adata.uns.keys())}')
print()
print(f'leiden categories ({adata.obs["leiden"].nunique()}):')
print(adata.obs['leiden'].value_counts().sort_index().to_string())
print()
print(f'dataset balance:\n{adata.obs["dataset"].value_counts().to_string()}')

## 2. Marker gene panels

Canonical marker sets per cell type, drawn from the cortical-development scRNA-seq literature (Polioudakis 2019, Bhaduri 2021, Pollen 2019, Trevino 2021). Panels are intentionally broad (3–6 genes per type) so that a panel mean is robust to dropout for any single gene.

**Cell types covered:**

| Type | Markers | Notes |
|---|---|---|
| **vRG** (ventricular radial glia) | VIM, SOX2, PAX6, HES1, GLI3, FABP7 | Cortical RG in the ventricular zone |
| **oRG** (outer/basal RG) | HOPX, MOXD1, FAM107A, PTPRZ1, TNC | Outer subventricular zone, primate-enriched |
| **Cycling G2/M** | MKI67, TOP2A, UBE2C, CCNB1, CDK1 | Mitotic progenitors |
| **Cycling S-phase** | PCNA, MCM2, MCM4, MCM7, CCNE2 | Replicating progenitors |
| **IPC** (intermediate progenitors) | EOMES, NEUROG1, NEUROG2, PPP1R17 | Cortical IPCs (TBR2+) |
| **Immature excitatory** | NEUROD2, NEUROD6, NEUROD4, STMN2 | Newborn cortical neurons |
| **Mature excitatory (deep layer)** | TBR1, BCL11B, FEZF2, FOXP2 | Layer 5/6 |
| **Mature excitatory (upper layer)** | SATB2, CUX2, POU3F2, BRN2 | Layer 2/3/4 |
| **GABAergic (general)** | GAD1, GAD2, DLX1, DLX2, DLX5, DLX6 | All cortical interneurons |
| **MGE-derived** | LHX6, NKX2-1, SST | Medial ganglionic eminence |
| **CGE-derived** | NR2F2, SP8, PROX1 | Caudal ganglionic eminence |
| **OPC** | OLIG1, OLIG2, PDGFRA, SOX10, NKX2-2 | Oligodendrocyte progenitors |
| **Astrocyte progenitors** | GFAP, AQP4, S100B, ALDH1L1, SLC1A3 | Astrocyte lineage |
| **Microglia** | AIF1, CX3CR1, P2RY12, C1QA, C1QB | Resident immune |
| **Vascular** | CLDN5, PECAM1, FLT1, KDR | Endothelium |
| **Choroid plexus** | TTR, KRT18, OTX2, FOLR1 | Off-target in organoids; **TTR is the specific marker** (KRT18/OTX2 are misleading alone) |
| **Stress / dying** | DDIT3, ATF3, GADD45A, NQO1, HSPA1A | Bhaduri 2020 stress signature |

### 2a — Define panels and validate against `.raw.var_names`

Build a dict mapping each cell type to its marker panel, then drop genes not present in the full 16,768-gene `.raw`. Panel scoring later only sees the present genes — drops are logged for transparency.

In [ ]:
PANELS = {
    'vRG':                  ['VIM', 'SOX2', 'PAX6', 'HES1', 'GLI3', 'FABP7'],
    'oRG':                  ['HOPX', 'MOXD1', 'FAM107A', 'PTPRZ1', 'TNC'],
    'Cycling_G2M':          ['MKI67', 'TOP2A', 'UBE2C', 'CCNB1', 'CDK1'],
    'Cycling_S':            ['PCNA', 'MCM2', 'MCM4', 'MCM7', 'CCNE2'],
    'IPC':                  ['EOMES', 'NEUROG1', 'NEUROG2', 'PPP1R17'],
    'Excitatory_immature':  ['NEUROD2', 'NEUROD6', 'NEUROD4', 'STMN2'],
    'Excitatory_deep':      ['TBR1', 'BCL11B', 'FEZF2', 'FOXP2'],
    'Excitatory_upper':     ['SATB2', 'CUX2', 'POU3F2'],
    'GABAergic':            ['GAD1', 'GAD2', 'DLX1', 'DLX2', 'DLX5', 'DLX6'],
    'MGE':                  ['LHX6', 'NKX2-1', 'SST'],
    'CGE':                  ['NR2F2', 'SP8', 'PROX1'],
    'OPC':                  ['OLIG1', 'OLIG2', 'PDGFRA', 'SOX10', 'NKX2-2'],
    'Astrocyte':            ['GFAP', 'AQP4', 'S100B', 'ALDH1L1', 'SLC1A3'],
    'Microglia':            ['AIF1', 'CX3CR1', 'P2RY12', 'C1QA', 'C1QB'],
    'Vascular':             ['CLDN5', 'PECAM1', 'FLT1', 'KDR'],
    'Choroid_plexus':       ['TTR', 'KRT18', 'OTX2', 'FOLR1'],
    'Stress':               ['DDIT3', 'ATF3', 'GADD45A', 'NQO1', 'HSPA1A'],
}

all_genes = set(adata.raw.var_names)
PANELS_FILTERED = {}
for cell_type, genes in PANELS.items():
    present = [g for g in genes if g in all_genes]
    missing = [g for g in genes if g not in all_genes]
    PANELS_FILTERED[cell_type] = present
    if missing:
        print(f'{cell_type:25}  present={len(present)}/{len(genes)}  missing: {missing}')
    else:
        print(f'{cell_type:25}  present={len(present)}/{len(genes)}')

## 3. Per-cluster panel scoring

**Caveat (from colab_09 §5a-caveat):** the panel mean is a *quick first-pass summary*, not a decision rule. A single dominant gene in a panel can drag the composite score and produce a wrong assignment (e.g. KRT18/OTX2 making cluster 0 look like choroid plexus when its actual top markers were cycling-RG genes VIM/TPI1/ENO1). Always cross-check against §4 unsupervised top markers before assigning a cluster.

### 3a — Compute per-cluster mean panel score

For each (cluster, panel) pair, take the mean expression of the panel's genes across cells in that cluster, on `.raw` (full-gene log-normalized). Output a `clusters × cell_types` dataframe.

In [ ]:
def panel_score_per_cluster(adata, panel_genes):
    if not panel_genes:
        return pd.Series(0.0, index=sorted(adata.obs['leiden'].unique(), key=int))
    idx = [adata.raw.var_names.get_loc(g) for g in panel_genes]
    X = adata.raw.X[:, idx]
    if hasattr(X, 'toarray'):
        X = X.toarray()
    score = X.mean(axis=1)
    df = pd.DataFrame({'leiden': adata.obs['leiden'].values, 'score': score})
    return df.groupby('leiden', observed=True)['score'].mean()

panel_scores = pd.DataFrame({
    cell_type: panel_score_per_cluster(adata, genes)
    for cell_type, genes in PANELS_FILTERED.items()
})
panel_scores.index = panel_scores.index.astype(str)
panel_scores = panel_scores.loc[sorted(panel_scores.index, key=int)]
print(panel_scores.round(2).to_string())

### 3b — Visualize panel scores as a heatmap

Clusters as rows, cell types as columns, color = mean panel score. Quick visual scan for which cluster lights up which panel. Use this to form *hypotheses* — confirm in §4 before assigning.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(panel_scores.values, aspect='auto', cmap='viridis')
ax.set_xticks(range(len(panel_scores.columns)))
ax.set_xticklabels(panel_scores.columns, rotation=45, ha='right')
ax.set_yticks(range(len(panel_scores.index)))
ax.set_yticklabels(panel_scores.index)
ax.set_xlabel('Cell-type panel')
ax.set_ylabel('Leiden cluster')
ax.set_title('Per-cluster mean expression of cell-type marker panels (panel mean — first-pass only, see §4)')
plt.colorbar(im, ax=ax, label='Mean panel expression (.raw)')
plt.tight_layout()
plt.savefig('panel_scores_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Unsupervised top markers per cluster (rank_genes_groups)

### 4a — Run `sc.tl.rank_genes_groups` on `leiden`, Wilcoxon

For each cluster, ranks genes by Wilcoxon test against all other clusters. Reads from `.raw` so we get full-gene resolution, not the 2,000 HVGs. This is the **authoritative** marker source for cluster identity.

In [ ]:
sc.tl.rank_genes_groups(
    adata, groupby='leiden',
    method='wilcoxon',
    use_raw=True,
    n_genes=25,
)
print('rank_genes_groups complete; top markers per cluster stored in adata.uns["rank_genes_groups"].')

### 4b — Print top 10 genes per cluster

Readable summary table. Each row is a cluster, each cell is its top-10 markers (Wilcoxon-ranked, reading from `.raw`).

In [ ]:
rgg = adata.uns['rank_genes_groups']
names = pd.DataFrame(rgg['names'])
scores = pd.DataFrame(rgg['scores'])

summary = pd.DataFrame({
    cl: [f'{names[cl].iloc[i]} ({scores[cl].iloc[i]:.0f})' for i in range(10)]
    for cl in sorted(names.columns, key=int)
}).T
summary.index.name = 'leiden'
summary.columns = [f'top{i+1}' for i in range(10)]

with pd.option_context('display.max_colwidth', 30, 'display.width', 300):
    print(summary.to_string())

### 4c — Dotplot of canonical markers across clusters

For each cluster, shows mean expression (color) and fraction of cells expressing (dot size) for a curated set of canonical lineage markers. This is the visual that lets you read off cell-type assignments at a glance — RG markers high in vRG/oRG clusters, EOMES in IPC clusters, GAD1/2 in interneuron clusters, etc.

In [ ]:
canonical_markers = [
    'VIM', 'SOX2', 'PAX6', 'HES1', 'FABP7',
    'HOPX', 'MOXD1', 'PTPRZ1',
    'MKI67', 'TOP2A', 'PCNA',
    'EOMES', 'NEUROG2',
    'NEUROD2', 'NEUROD6', 'STMN2',
    'TBR1', 'BCL11B', 'FEZF2',
    'SATB2', 'CUX2',
    'GAD1', 'GAD2', 'DLX2', 'DLX5',
    'LHX6', 'NR2F2',
    'OLIG1', 'PDGFRA',
    'GFAP', 'AQP4',
    'AIF1', 'C1QA',
    'CLDN5', 'PECAM1',
    'TTR',
    'DDIT3', 'NQO1',
]
canonical_present = [g for g in canonical_markers if g in adata.raw.var_names]

sc.pl.dotplot(
    adata, var_names=canonical_present, groupby='leiden',
    use_raw=True, standard_scale='var',
    save='_canonical_markers_by_leiden.png', show=True,
)

## 5. Build cluster → cell-type mapping

After inspecting §3 heatmap, §4b top-10 marker table, and §4c dotplot, fill in the dict below. Each Leiden cluster (string keys '0' through '20') maps to a cell-type label. Use the panel names from §2 as the canonical labels (e.g. `'vRG'`, `'IPC'`, `'Excitatory_deep'`, `'GABAergic_MGE'`, `'Choroid_plexus'`).

Add suffixes when biology demands it: `'vRG_organoid'` if a cluster is organoid-pure RG (the cluster-0 case from colab_09 — 98.7% Bhaduri 2020), `'Stressed'` for stress-signature-dominant clusters, etc.

**Reference from prior runs** (colab_03 / colab_04, different clustering — *not* directly applicable to colab_08's 21 clusters, only as a rough prior):
- vRG, mature/immature/maturing excitatory, oRG variants, GABAergic interneurons, cycling progenitors at multiple cell-cycle stages, astrocyte progenitors, stress, choroid plexus.
- No microglia or OPC consolidation expected on the organoid side; both should appear on the fetal side.

### 5a — Cluster → cell-type dict (FILL IN AFTER §3/§4)

Initial values are TODO placeholders. Update each entry based on the marker evidence, then run the rest of the notebook. If you want to leave clusters unannotated, use `'unassigned'` or split a single label into two suffixed labels.

In [ ]:
cluster_to_celltype = {
    '0':  'TODO',
    '1':  'TODO',
    '2':  'TODO',
    '3':  'TODO',
    '4':  'TODO',
    '5':  'TODO',
    '6':  'TODO',
    '7':  'TODO',
    '8':  'TODO',
    '9':  'TODO',
    '10': 'TODO',
    '11': 'TODO',
    '12': 'TODO',
    '13': 'TODO',
    '14': 'TODO',
    '15': 'TODO',
    '16': 'TODO',
    '17': 'TODO',
    '18': 'TODO',
    '19': 'TODO',
    '20': 'TODO',
}

actual_clusters = set(adata.obs['leiden'].astype(str).unique())
mapped_clusters = set(cluster_to_celltype.keys())
missing = actual_clusters - mapped_clusters
extra   = mapped_clusters - actual_clusters
if missing:
    print(f'WARN: clusters present in data but missing from map: {sorted(missing, key=int)}')
if extra:
    print(f'WARN: clusters in map but not in data: {sorted(extra, key=int)}')
if not missing and not extra:
    print(f'OK: all {len(actual_clusters)} clusters mapped.')

todo_count = sum(1 for v in cluster_to_celltype.values() if v == 'TODO')
if todo_count:
    print(f'\nWARN: {todo_count} clusters still set to TODO — fill them in before continuing.')

## 6. Apply mapping

### 6a — Add `cell_type` column on obs and order the categories

Apply the dict, cast to ordered categorical with cell types sorted by lineage (progenitors → cycling → IPC → immature → mature neurons → GABAergic → glia → off-target/stress) so downstream plots have a sensible legend order.

In [ ]:
adata.obs['cell_type'] = adata.obs['leiden'].astype(str).map(cluster_to_celltype)

if adata.obs['cell_type'].isna().any():
    n_nan = adata.obs['cell_type'].isna().sum()
    raise RuntimeError(f'{n_nan} cells have no cell_type assignment — check the mapping dict')

lineage_order = [
    'vRG', 'vRG_organoid', 'vRG_fetal',
    'oRG', 'oRG_organoid', 'oRG_fetal',
    'Cycling_S', 'Cycling_G2M', 'Cycling',
    'IPC',
    'Excitatory_immature', 'Excitatory_maturing',
    'Excitatory_deep', 'Excitatory_upper', 'Excitatory_mature',
    'GABAergic', 'GABAergic_MGE', 'GABAergic_CGE',
    'OPC', 'Astrocyte',
    'Microglia', 'Vascular',
    'Choroid_plexus', 'Stressed',
    'unassigned',
]
present_in_data = [t for t in lineage_order if t in adata.obs['cell_type'].unique()]
extra = [t for t in adata.obs['cell_type'].unique() if t not in lineage_order]
final_order = present_in_data + sorted(extra)
adata.obs['cell_type'] = pd.Categorical(adata.obs['cell_type'], categories=final_order, ordered=True)

print(adata.obs['cell_type'].value_counts(sort=False).to_string())

## 7. UMAP visualisations

### 7a — UMAP coloured by cell_type

The canonical map. Compare against the §4c dotplot to confirm the assignment is internally consistent.

In [ ]:
sc.pl.umap(
    adata, color='cell_type',
    legend_loc='right margin',
    frameon=False, size=8,
    save='_cell_type.png', show=True,
)

### 7b — Per-cell-type UMAPs (full-size individuals)

One UMAP per cell type, with that type highlighted and the rest in light grey. Honors the `feedback_plots_individual` rule — when a multi-class plot is too crowded to read accurately, individual full-size plots are required for verification.

Path of the per-type plots: `figures/umap_by_celltype_<TYPE>.png`.

In [ ]:
for ct in adata.obs['cell_type'].cat.categories:
    if (adata.obs['cell_type'] == ct).sum() == 0:
        continue
    sc.pl.umap(
        adata, color='cell_type', groups=[ct],
        frameon=False, size=8, legend_loc='right margin',
        title=f'cell_type = {ct}  (n = {(adata.obs["cell_type"] == ct).sum():,})',
        save=f'_by_celltype_{ct}.png', show=True,
    )

## 8. Composition diagnostics

### 8a — cell_type × dataset crosstab (counts and row-normalized)

For each cell type, what fraction is organoid vs fetal? Cell types that are ca. 50/50 indicate Harmony successfully aligned that lineage. Cell types that are >95% one-side either reflect biology (microglia, OPC = fetal-only; choroid plexus, stress = organoid-only) or mark the integration limitations.

In [ ]:
ct_counts = pd.crosstab(adata.obs['cell_type'], adata.obs['dataset'])
ct_pct    = pd.crosstab(adata.obs['cell_type'], adata.obs['dataset'], normalize='index') * 100
ct_pct    = ct_pct.round(1)
ct_pct['total'] = ct_counts.sum(axis=1)
print('cell_type × dataset (counts):')
print(ct_counts.to_string())
print()
print('cell_type × dataset (% of cell type, row-normalized):')
print(ct_pct.to_string())

### 8b — Stacked bar: cell-type composition per dataset

For each dataset, the proportion of each cell type. Tells you the overall composition difference between organoid and fetal at a glance — confirms expected patterns (organoids progenitor-heavy, fetal neuron-heavy, GABAergic surge in fetal).

In [ ]:
ds_pct = pd.crosstab(adata.obs['dataset'], adata.obs['cell_type'], normalize='index') * 100
ds_pct = ds_pct.round(2)

fig, ax = plt.subplots(figsize=(12, 5))
ds_pct.plot(kind='barh', stacked=True, ax=ax, colormap='tab20', width=0.7)
ax.set_xlabel('% of dataset')
ax.set_ylabel('Dataset')
ax.set_title('Cell-type composition by dataset')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.tight_layout()
plt.savefig('composition_by_dataset.png', dpi=120, bbox_inches='tight')
plt.show()

### 8c — Cell-type composition by gestational age (fetal) and protocol week (organoid)

If `age_gw` (fetal) and `age_week` (organoid) survived concat with NaN-fill, plot per-age cell-type composition for each. This is the developmental progression view — immature/cycling at early ages, mature neurons/GABAergic later. Skipped silently if either column is absent or all-NaN.

In [ ]:
for age_col, ds_label in [('age_gw', 'bhaduri_2021'), ('age_week', 'bhaduri_2020')]:
    if age_col not in adata.obs.columns:
        print(f'(skip) {age_col} not in obs')
        continue
    sub = adata.obs[adata.obs['dataset'] == ds_label]
    sub = sub.dropna(subset=[age_col])
    if len(sub) == 0:
        print(f'(skip) {age_col} all-NaN for {ds_label}')
        continue
    age_pct = pd.crosstab(sub[age_col], sub['cell_type'], normalize='index') * 100
    fig, ax = plt.subplots(figsize=(12, 4))
    age_pct.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', width=0.85)
    ax.set_ylabel('% of cells at age')
    ax.set_title(f'{ds_label}: cell-type composition by {age_col}')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7)
    plt.tight_layout()
    plt.savefig(f'composition_{ds_label}_by_{age_col}.png', dpi=120, bbox_inches='tight')
    plt.show()

## 9. Save annotated object

### 9a — Write `integrated_annotated_100k.h5ad` to Drive

Final output. Adds `obs['cell_type']` to the colab_08 integrated object; everything else is propagated unchanged. This is the input to colab_05's rerun (trajectory analysis).

In [ ]:
adata.write_h5ad(PATHS['annotated_out'])
size_gb = os.path.getsize(PATHS['annotated_out']) / 1e9
print(f'Saved: {PATHS["annotated_out"]}')
print(f'Size:  {size_gb:.2f} GB')
print(f'Final shape: {adata.shape}')
print(f'Cell types ({adata.obs["cell_type"].nunique()}):')
print(adata.obs['cell_type'].value_counts().to_string())